# Getting Started with GitOps Agent Development

Welcome to the Firewall Automation Hackathon! This notebook will guide you through building the GitOps Agent as a standalone agent.

## Overview

The GitOps Agent is a **standalone specialist agent** that handles all firewall rule generation and Git operations. It will be invoked by the supervisor agent when users request firewall rule changes. This agent:
- Clones the SecOps Git repository from Azure DevOps
- Reads existing Suricata firewall rules
- Generates new Suricata rules with proper metadata
- Creates Git branches with naming conventions
- Commits and pushes rule changes
- Creates pull requests with full metadata (Phase 2)

## Development Phases

**Phase 1 (FWAUTO-14)**: Git operations and rule generation
- Clone repository
- Read existing rules
- Generate Suricata rules
- Create branches, commit, and push

**Phase 2 (FWAUTO-19)**: PR automation
- Create pull requests via Azure DevOps REST API
- Fill out firewall change request document templates
- Assign reviewers based on rule risk level

## How to Get Started

Your task is to create a **new standalone agent** that uses [Strands Agents SDK](https://strandsagents.com/latest/) and the GitPython library to handle Git operations.

### Authentication Setup

Azure DevOps access requires a Personal Access Token (PAT):
- **PAT stored in**: AWS Secrets Manager (`firewall-chatbot/azure-devops/pat`)
- The agent will retrieve the PAT from Secrets Manager for Git authentication
- I have setup a fork of the SecOps repo for this hackathon. It is available here: https://pace-devops.visualstudio.com/_git/IaC-AWS-Firewall-Automation

### Key Libraries

- **GitPython**: For Git operations (clone, branch, commit, push)
- **Azure DevOps REST API**: For PR creation (Phase 2)
- **boto3**: For Secrets Manager access

## Related Jira Tasks

- [FWAUTO-14: Firewall GitSecOps agent implementation (Phase 1)](https://your-jira-instance.atlassian.net/browse/FWAUTO-14)
- [FWAUTO-19: Enhance GitSecOps Agent - PR creation and automation (Phase 2)](https://your-jira-instance.atlassian.net/browse/FWAUTO-19)

## Step 1: Copy the Base Agent Code

Copy the supervisor agent code to your workspace so you can modify it.

In [2]:
%%bash
# Copy the base agent code from the repository
cp -r /home/sagemaker-user/IST-AWS-Firewall-Automation/agent .

In [2]:
# View the base agent python code
with open('agent/src/github-agent.py', 'r') as f:
    print(f.read())

"""
ADO Agent - AI-powered Git automation using AWS Bedrock AgentCore.

This agent can clone repositories, make changes, commit, push, and create pull requests
automatically using natural language instructions.
"""

from pprint import pp
import os
import boto3
import subprocess
import tempfile
from pathlib import Path
import json

import requests
from strands import Agent
from strands.models import BedrockModel
from strands.tools import tool
from strands_tools import editor, file_read, file_write

from bedrock_agentcore.runtime import BedrockAgentCoreApp

# Initialize AgentCore application
app = BedrockAgentCoreApp()

# Bypass tool consent for automated execution
os.environ["BYPASS_TOOL_CONSENT"] = "true"

# Disable editor tool creating .bak files
os.environ["EDITOR_DISABLE_BACKUP"] = "true"

# TODO Secrets manager
def get_secret(secret_name: str) -> str:
    client = boto3.client('secretsmanager')
    response = client.get_secret_value(SecretId=secret_name)
    return response['Secret

## Step 2: Review the README

Study the README.md file in this folder to understand the agent structure and tool definitions needed.

In [ ]:
# View the README for implementation guidance
with open('README.md', 'r') as f:
    print(f.read())

## Step 3: Create Your GitOps Agent

Create a new standalone agent for Git operations and firewall rule generation. This agent will run separately and be invoked by the supervisor agent.

### Key Implementation Steps:

1. **Create a new agent file** - Start with `gitops_agent.py`
2. **Set up Azure DevOps authentication** - Retrieve PAT from AWS Secrets Manager
3. **Implement Git tools** - Define tools for Git operations:
   - `clone_firewall_rules_repo()` - Clone SecOps repo from Azure DevOps
   - `read_existing_rules()` - Parse and read existing Suricata rules
   - `generate_suricata_rule()` - Generate new rules with metadata
   - `create_feature_branch()` - Create branch using naming convention
   - `commit_and_push_rule()` - Commit and push changes
   - `create_pull_request()` - Create PR via Azure DevOps API (Phase 2)
4. **Add validation** - Validate Suricata rule syntax and check for duplicates
5. **Add error handling** - Handle Git errors, authentication failures, network issues
6. **Implement AgentCore entrypoint** - Set up the async streaming entrypoint

### Suricata Rule Format Example:

```
# Metadata
# SRA: SRA-12345
# Account: RT-Prod (123456789012)
# Justification: Allow API access for prod web servers
# Created: 2025-01-15
# Created-By: Firewall Automation Chatbot

pass tls $HOME_NET any -> $EXTERNAL_NET 443 (msg:"Allow HTTPS to api.example.com"; tls.sni; content:"api.example.com"; sid:1000001; rev:1;)
```

### Branch Naming Convention:

```
firewall/allow-tls-rtprod-apiexample
firewall/drop-http-rtdev-suspicious
```

### Implementation Approach:

This is a **specialist agent** focused solely on Git operations and rule generation:
- Uses GitPython library for Git operations (clone, branch, commit, push)
- Generates Suricata rules following the organization's format
- Validates rule syntax before committing
- Checks for duplicate rules
- Creates detailed commit messages with metadata
- The supervisor agent will invoke this agent when users request rule changes

In [4]:
# View the current dummy implementation
with open('agent/src/agent.py', 'r') as f:
    content = f.read()
    # Find and display the execute_git_operation function
    start = content.find('def execute_git_operation')
    end = content.find('\n\n@tool', start)
    if start != -1:
        print(content[start:end if end != -1 else start+1000])

def execute_git_operation(
    operation: str,
    rule_content: str = None,
    branch_name: str = None,
    commit_message: str = None
):
    """
    Execute Git operations for firewall rule management.

    Args:
        operation: Git operation to perform (clone, branch, commit, push, create_pr)
        rule_content: Suricata rule content (for commits)
        branch_name: Branch name for operations
        commit_message: Commit message

    Returns:
        Result of the Git operation
    """
    return {
        "operation": operation,
        "status": "success",
        "branch": branch_name or "firewall/add-tcp-production",
        "commit_sha": "a1b2c3d4e5f6",
        "message": f"Git operation '{operation}' completed successfully",
        "pr_url": "https://dev.azure.com/org/project/_git/firewall-rules/pullrequest/42" if operation == "create_pr" else None
    }



## Step 4: Test Your Implementation

Test the tool locally before deploying.

In [2]:
# TODO: Add your test code here
# Example test:
from agent.src.git_agent import clone_repo

# Test cloning the repo
repo_result = clone_repo("main")

# Test generating a rule
# rule = generate_suricata_rule(
#     action="pass",
#     protocol="tls",
#     source="10.100.0.0/16",
#     destination="api.example.com",
#     port=443,
#     metadata={
#         "sra": "SRA-12345",
#         "account": "RT-Prod",
#         "account_id": "123456789012",
#         "justification": "Allow API access for prod web servers"
#     }
# )
# print(f"Generated rule:\n{rule['rule_content']}")

/tmp/tmpoyf8t8oq/network-firewall


Cloning into '/tmp/tmpoyf8t8oq/network-firewall'...


Repository cloned to: /tmp/tmpoyf8t8oq/network-firewall


Switched to a new branch 'main'


## Step 5: Deploy to AWS

Deploy your updated agent to AWS Bedrock AgentCore Runtime.

In [10]:
import boto3
import os
os.environ["AWS_DEFAULT_REGION"] = "ap-southeast-2"
os.environ["AWS_STS_REGIONAL_ENDPOINTS"] = "regional"

In [5]:
import time
import boto3
from bedrock_agentcore_starter_toolkit import Runtime

# Initialize the runtime toolkit
region = "ap-southeast-2"

agentcore_runtime = Runtime()

# Configure the deployment
response = agentcore_runtime.configure(
    agent_name="gitops_tool",  # TODO: Set your agent name, e.g., "firewall-gitops-agent"
    entrypoint="agent/src/agent.py",  # TODO: Set your entrypoint file, e.g., "gitops_agent.py"
    execution_role="arn:aws:iam::123456789012:role/YourAgentCoreExecutionRole",
    code_build_execution_role="arn:aws:iam::123456789012:role/YourCodeBuildRole",
    auto_create_ecr=True,
    requirements_file="agent/src/requirements.txt",  # TODO: Set your requirements file, e.g., "requirements.txt"
    region=region,
    memory_mode="STM_ONLY",
)

print("Configuration completed:", response)

launch_result = agentcore_runtime.launch()
print("Launch completed:", launch_result.agent_arn)

# Wait for the agent to be ready
status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]

end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    print(f"Waiting for deployment... Current status: {status}")
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]

if status == "READY":
    runtime_id = status_response.agent["agentRuntimeId"]

    # Update the runtime to be deployed in VPC
    client = boto3.client("bedrock-agentcore-control", region_name=region)

    response = client.update_agent_runtime(
        agentRuntimeId=runtime_id,
        networkConfiguration={
            "networkMode": "VPC",
            "networkModeConfig": {
                "subnets": ["subnet-xxxxxxxxxxxxxxxxx", "subnet-yyyyyyyyyyyyyyyyy"],
                "securityGroups": ["sg-xxxxxxxxxxxxxxxxx"],
            },
        },
        agentRuntimeArtifact={
            "containerConfiguration": {
                "containerUri": "123456789012.dkr.ecr.ap-southeast-2.amazonaws.com/firewall-automation/gitops_tool:latest"  # TODO: Set your container URI, e.g., "123456789012.dkr.ecr.ap-southeast-2.amazonaws.com/firewall-automation/gitops-agent:latest"
            }
        },
        roleArn="arn:aws:iam::123456789012:role/YourAgentCoreExecutionRole",
    )
    
print(f"GitOps Agent deployed successfully!")

Entrypoint parsed: file=/home/sagemaker-user/your-username/IST-AWS-Firewall-Automation/notebooks/gitops-agent/agent/src/agent.py, bedrock_agentcore_name=agent
Memory configured with STM only
Configuring BedrockAgentCore agent: gitops_tool
Will create new memory with mode: STM_ONLY
Memory configuration: Short-term memory only
Found existing memory ID from previous launch: gitops_tool_mem-J4B78VBodA


⚠️ Platform mismatch: Current system is 'linux/amd64' but Bedrock AgentCore requires 'linux/arm64', so local builds
won't work.
Please use default launch command which will do a remote cross-platform build using code build.For deployment other
options and workarounds, see: 
https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html

Generated Dockerfile: Dockerfile
Generated .dockerignore: /home/sagemaker-user/your-username/IST-AWS-Firewall-Automation/notebooks/gitops-agent/.dockerignore
Keeping 'gitops_tool' as default agent
Bedrock AgentCore configured: /home/sagemaker-user/your-username/IST-AWS-Firewall-Automation/notebooks/gitops-agent/.bedrock_agentcore.yaml
🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containers in the cloud with CodeBuild
   • No local Docker required
💡 Available deployment modes:
   • runtime.launch()                           → CodeBuild (current)
   • runtime.launch(local=True)                 → Local development
   • runtime.launch(local_build=True)           → Local build + cloud deploy (NEW)
Creating memory resource for agent: gitops_tool
✅ MemoryManager initialized for region: ap-southeast-2


Configuration completed: config_path=PosixPath('/home/sagemaker-user/your-username/IST-AWS-Firewall-Automation/notebooks/gitops-agent/.bedrock_agentcore.yaml') dockerfile_path=PosixPath('/home/sagemaker-user/your-username/IST-AWS-Firewall-Automation/notebooks/gitops-agent/Dockerfile') dockerignore_path=PosixPath('/home/sagemaker-user/your-username/IST-AWS-Firewall-Automation/notebooks/gitops-agent/.dockerignore') runtime='Docker' region='ap-southeast-2' account_id='123456789012' execution_role='arn:aws:iam::123456789012:role/YourAgentCoreExecutionRole' ecr_repository=None auto_create_ecr=True memory_id=None


🔎 Retrieving memory resource with ID: gitops_tool_mem-J4B78VBodA...
  Found memory: gitops_tool_mem-J4B78VBodA
Found existing memory in cloud: gitops_tool_mem-J4B78VBodA
Existing memory has 0 strategies
✅ Using existing STM-only memory
Starting CodeBuild ARM64 deployment for agent 'gitops_tool' to account 123456789012 (ap-southeast-2)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: gitops_tool
✅ ECR repository available: 123456789012.dkr.ecr.ap-southeast-2.amazonaws.com/bedrock-agentcore-gitops_tool
Using execution role from config: arn:aws:iam::123456789012:role/YourAgentCoreExecutionRole
Preparing CodeBuild project and uploading source...
Using CodeBuild role from config: arn:aws:iam::123456789012:role/YourCodeBuildRole


✅ Reusing existing ECR repository: 123456789012.dkr.ecr.ap-southeast-2.amazonaws.com/bedrock-agentcore-gitops_tool


Using dockerignore.template with 45 patterns for zip filtering
Uploaded source to S3: gitops_tool/source.zip
Updated CodeBuild project: bedrock-agentcore-gitops_tool-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.0s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 9.3s
🔄 DOWNLOAD_SOURCE started (total: 10s)
✅ DOWNLOAD_SOURCE completed in 2.1s
🔄 BUILD started (total: 12s)
✅ BUILD completed in 19.6s
🔄 POST_BUILD started (total: 32s)
✅ POST_BUILD completed in 9.3s
🔄 COMPLETED started (total: 41s)
✅ COMPLETED completed in 1.0s
🎉 CodeBuild completed successfully in 0m 42s
CodeBuild completed successfully
✅ CodeBuild project configuration saved
Deploying to Bedrock AgentCore...
Passing memory configuration to agent: gitops_tool_mem-J4B78VBodA
✅ Agent created/updated: arn:aws:bedrock-agentcore:ap-southeast-2:123456789012:runtime/gitops_tool-5Ld8h3FhNi
Observability is enabl

Launch completed: arn:aws:bedrock-agentcore:ap-southeast-2:123456789012:runtime/gitops_tool-5Ld8h3FhNi


🔎 Retrieving memory resource with ID: gitops_tool_mem-J4B78VBodA...
  Found memory: gitops_tool_mem-J4B78VBodA
Retrieved Bedrock AgentCore status for: gitops_tool


GitOps Agent deployed successfully!


## Step 6: Test End-to-End

Test the deployed agent through the Streamlit UI.

## Step 7: Push Your Changes

Create a branch and push your changes for review.

## Tips and Best Practices

### Security
- **Store Azure DevOps PAT** in AWS Secrets Manager, never hardcode
- **Limit PAT scope**: Only Code (read/write) and Pull Requests (read/write)
- **No admin permissions**: No delete, force push, or admin access
- **Automatic rotation**: 90-day expiration with CloudWatch alerts
- **Audit logging**: Log all Git operations with user context
- **Ephemeral storage**: Clean up `/tmp` directories after each operation

### Git Operations
- **Clone to temporary directories**: Use `/tmp/firewall-rules-{uuid}` for isolation
- **Branch naming**: Follow convention `firewall/{action}-{protocol}-{account}`
- **Commit messages**: Include full metadata (SRA, account, justification)
- **Error handling**: Handle merge conflicts, authentication failures, network issues
- **Cleanup**: Always remove cloned repositories after operations complete

### Rule Generation
- **Validate syntax**: Check Suricata rule format before committing
- **Check duplicates**: Read existing rules to prevent duplicates
- **Metadata format**: Follow organization's metadata comment format
- **SID management**: Ensure unique SIDs (coordinate with security team)
- **Test rules**: Validate rules can be parsed by Suricata

### User Experience
- **Progress updates**: Show "Cloning repository...", "Generating rule...", etc.
- **Clear error messages**: Explain what went wrong and how to fix it
- **PR links**: Return PR URL so users can track their request
- **Confirmation**: Show rule content before committing

### Performance
- **Shallow clones**: Use `--depth 1` for faster cloning
- **Parallel operations**: Don't block on long-running Git operations
- **Timeout handling**: Set reasonable timeouts for Git operations
- **Connection pooling**: Reuse Git connections when possible

## Resources

- README: `README.md` in this folder
- Jira Phase 1: [FWAUTO-14](https://your-jira-instance.atlassian.net/browse/FWAUTO-14)
- Jira Phase 2: [FWAUTO-19](https://your-jira-instance.atlassian.net/browse/FWAUTO-19)
- GitPython Documentation: https://gitpython.readthedocs.io
- Azure DevOps REST API: https://learn.microsoft.com/en-us/rest/api/azure/devops/
- Suricata Rule Syntax: https://docs.suricata.io/en/latest/rules/